## Simulating the 2D Time-Dependent Schrödinger Equation (TDSE)

#### Rafał Głodek, Antoni Krzak, Michał Sapijaszko

---

### **I Physical and mathematical foundations**

#### **The governing equation**
We will be numerically integrating the 2D TDSE:

$$
i\hbar \frac{\partial \psi(x,y,t)}{\partial t} = \left( -\frac{\hbar^2}{2m} \nabla^2 + V(x,y) \right) \psi(x,y,t)
$$.

To simplify our computational model, we will use natural units (where $\hbar = 1$ and $m = 1$), reducing the constants and avoiding floating-point errors.

### **The initial state**
The particle will be represented by a 2D Gaussian wave packet with an initial momentum $k_0$ in the $y$-direction:
$$
\psi(x,y,0) = A \exp\left(-\frac{(x-x_0)^2 + (y-y_0)^2}{2\sigma^2} + i k_0 y\right)
$$

### **The potential landscapes ($V(x,y)$)**
* Double-slit: An infinitely high potential barrier spanning the $x$-axis, interrupted by two narrow gaps (regions where $V = 0$).
* Tunneling barrier: A finite rectangular potential barrier of height $V_0$ and thickness $d$. Crucially, we will set the packet's kinetic energy $E$ such that $E < V_0$ to observe the purely quantum mechanical "leakage."

### **II Numerical methodology**

#### Crank-Nicolson method (finite difference):
* Pros: Unconditionally stable, preserves total probability (unitarity).
* Cons: Requires solving a massive system of linear equations at each time step. In 2D, this becomes computationally expensive very quickly, even with sparse matrices.

#### Split-Operator Fourier Transform (SOFT) method:
* Pros: Exceptionally fast. It splits the Hamiltonian into kinetic and potential operators. The kinetic part is solved trivially in momentum space using the Fast Fourier Transform (FFT), while the potential part is solved in real space.
* Cons: Conditionally stable (requires careful tuning of the time step $\Delta t$ relative to the spatial grid $\Delta x$).
* Our Choice: We will use the SOFT Method. It is the industry standard for propagating 2D wave packets because Python's scipy.fft module is highly optimized.

### **III Code architecture**

1. Grid: A class to initialize the 2D spatial domain ($x$ and $y$ arrays) and the reciprocal momentum space grids ($k_x$ and $k_y$).
2. Potentials: Functions to generate 2D arrays representing our different scenarios (walls, slits, and finite barriers).
3. WavePacket: A class to initialize the Gaussian wave function and manage its state array.
4. Propagator: The core physics engine. A function that takes the current $\psi$, applies the half-step potential operator, performs an FFT, applies the kinetic operator, performs an inverse FFT, and applies the final half-step potential operator.

### **IV Visualization and analysis**
Since our primary goal includes smooth animations of complex phenomena, the visualization layer is just as critical as the physics engine.
1. Probability Density: The simulation outputs complex numbers. We will visualize the probability density, which is the absolute square of the wave function: $P(x,y,t) = |\psi(x,y,t)|^2$.
2. Animation Engine: We will use matplotlib.animation.FuncAnimation to iteratively call our propagator and update an imshow plot. We will use a perceptually uniform colormap (like viridis or magma) to accurately represent the intensity of the wave.
3. Tunneling Metrics: For the tunneling scenario, we will add an integration function to calculate the total probability of finding the particle on the "far side" of the barrier as time progresses, proving that transmission occurred.